# 23.1 车载与功能安全（仿真清单）

演示：LLM 与安全链隔离、推理看门狗、超时进入安全态。

In [ ]:
import hashlib
import json
import math
import time
from dataclasses import dataclass, field
from typing import Optional
import numpy as np

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    torch.manual_seed(42)
    print(f"PyTorch {torch.__version__}")
except Exception as e:
    torch = None
    print("torch unavailable:", e)

np.random.seed(42)

In [ ]:
class Watchdog:
    def __init__(self, limit_ms):
        self.limit_ms = limit_ms

    def run(self, fn, *args, **kwargs):
        t0 = time.perf_counter()
        try:
            out = fn(*args, **kwargs)
        except Exception as e:
            return {"safe_state": True, "error": str(e)}
        dt = (time.perf_counter() - t0) * 1000
        if dt > self.limit_ms:
            return {"safe_state": True, "reason": "deadline_exceeded", "dt": dt}
        return {"safe_state": False, "result": out, "dt": dt}


def llm_suggest_route(query):
    time.sleep(0.02)
    return "建议前往最近服务区"


def unsafe_actuator_brake():
    raise RuntimeError("LLM must not call brake")


wd = Watchdog(50)
print(wd.run(llm_suggest_route, "我累了"))
print(wd.run(lambda: time.sleep(0.08) or "late"))


def mediate(intent, llm_text):
    # 安全相关意图不执行 LLM 建议
    if intent in {"brake", "steer", "accelerate"}:
        return {"action": "reject", "hmi": "该操作不受语音模型控制"}
    return {"action": "hmi_suggest", "text": llm_text}

print(mediate("nav_chat", "前面右转"))
print(mediate("brake", "帮我刹车"))

## 小结

LLM 只做建议与信息娱乐；安全链隔离；超时必须进安全态；变更受安全论证约束。